# PHASE 0: QWEN IMAGE 2.1 MODEL EXPLORATION

In [1]:
import torch
from diffusers import QwenImage21Pipeline

pipe = QwenImage21Pipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1", torch_dtype=torch.bfloat16
)

c:\Users\jjmca\nihonga\.venv\Lib\site-packages\diffusers\utils\deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## PIPELINE COMPONENTS
- A VLM to process both prompt and image references
- A scheduler, to guide denoising proecss
- MMDiTs x 32 (Multimodal Diffusion transformers)
- VAE Encoders to encode the image latents over the denoising process

In [2]:
print(pipe)

QwenImage21Pipeline {
  "_class_name": "QwenImage21Pipeline",
  "_diffusers_version": "0.41.0.dev0",
  "_name_or_path": "Qwen/Qwen-Image-2.1",
  "processor": [
    "transformers",
    "Qwen3VLProcessor"
  ],
  "scheduler": [
    "diffusers",
    "FlowMatchEulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "Qwen3VLForConditionalGeneration"
  ],
  "transformer": [
    "diffusers",
    "QwenImage21Transformer2DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKLQwenImage21"
  ]
}



In [3]:
print(pipe.transformer)

QwenImage21Transformer2DModel(
  (pos_embed): QwenImage21Rope()
  (time_text_embed): QwenImage21TimestepProjEmbeddings(
    (time_proj): QwenImage21TemporalTimesteps()
    (timestep_embedder): TimestepEmbedding(
      (linear_1): Linear(in_features=256, out_features=4096, bias=False)
      (act): SiLU()
      (linear_2): Linear(in_features=4096, out_features=4096, bias=False)
    )
  )
  (txt_in): QwenImage21TextProjection(
    (text_norm): QwenImage21ZeroCenterRMSNorm()
    (in_layer): Linear(in_features=4096, out_features=4096, bias=False)
    (act): GELU(approximate='tanh')
    (out_layer): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (img_in): Linear(in_features=64, out_features=4096, bias=False)
  (modulation): Sequential(
    (0): SiLU()
    (1): Linear(in_features=4096, out_features=16384, bias=False)
  )
  (transformer_blocks): ModuleList(
    (0-31): 32 x QwenImage21TransformerBlock(
      (img_norm1): LayerNorm((4096,), eps=1e-06, elementwise_affine=False, bi

```
                                   QWEN-IMAGE 2.1
                         QwenImage21Transformer2DModel
                                      │
        ┌─────────────────────────────┼─────────────────────────────┐
        │                             │                             │
        ▼                             ▼                             ▼
  TEXT / CONTEXT                CURRENT LATENT x_t              TIMESTEP t
  from Qwen VLM                 # current image state           # how far we are
  # prompt / refs               # starts as pure noise          # in denoising
                                # becomes cleaner over steps
        │                             │                             │
        │                             │                             ▼
        │                             │                 ┌─────────────────────────┐
        │                             │                 │ TemporalTimesteps       │
        │                             │                 │ scalar t → 256          │
        │                             │                 │ # encode noise level    │
        │                             │                 └────────────┬────────────┘
        │                             │                              │
        ▼                             ▼                              ▼
┌──────────────────────┐   ┌──────────────────────┐      ┌─────────────────────────┐
│ Text Projection      │   │ img_in               │      │ TimestepEmbedding       │
│                      │   │ Linear 64 → 4096     │      │                         │
│ RMSNorm              │   │ # current latent     │      │ Linear 256 → 4096       │
│ # stabilize scale    │   │   has 64 features    │      │ # map time encoding     │
│                      │   │   per latent token   │      │   to model width        │
│ Linear 4096 → 4096   │   │                      │      │                         │
│ GELU                 │   │ # expand it into     │      │ SiLU                    │
│ Linear 4096 → 4096   │   │   4096-D token       │      │ # nonlinearity          │
│ # adapt VLM features │   └──────────┬───────────┘      │                         │
└──────────┬───────────┘              │                  │ Linear 4096 → 4096      │
           │                          │                  │ # final timestep vector │
           │                          │                  └────────────┬────────────┘
           │                          │                               │
           │                          │                               ▼
           │                          │                  ┌─────────────────────────┐
           │                          │                  │ Modulation              │
           │                          │                  │                         │
           │                          │                  │ SiLU                    │
           │                          │                  │ Linear 4096 → 16384     │
           │                          │                  │ # turn timestep into    │
           │                          │                  │   control values        │
           │                          │                  │ # controls norm/gates   │
           │                          │                  └────────────┬────────────┘
           │                          │                               │
           ▼                          ▼                               │
   text tokens                   image tokens                         │
   N_txt × 4096                  N_img × 4096                         │
   # what to draw                # current image state                │
           │                          │                               │
           └──────────────┬───────────┘                               │
                          │                                           │
                          ▼                                           │
                 ┌─────────────────────┐                              │
                 │   JOINT SEQUENCE    │◄─────────────────────────────┘
                 │ [text][image ...]   │
                 │ # same hidden size  │
                 │ # processed together│
                 └──────────┬──────────┘
                            │
                            ▼
        ╔══════════════════════════════════════════════╗
        ║          TRANSFORMER BLOCK × 32              ║
        ║          hidden size = 4096                  ║
        ╚══════════════════════════════════════════════╝
                            │
                            ▼
                 ┌──────────────────────┐
                 │ LayerNorm #1         │
                 │ # normalize token    │
                 │   values so scales   │
                 │   stay stable        │
                 └──────────┬───────────┘
                            │
                            │ + timestep modulation
                            │ # behavior changes with t
                            ▼
                 ┌──────────────────────┐
                 │ Q / K / V            │
                 │ projections          │
                 │                      │
                 │ Q: 4096 → 4096       │
                 │ # what token seeks   │
                 │                      │
                 │ K: 4096 → 4096       │
                 │ # what token offers  │
                 │   for matching       │
                 │                      │
                 │ V: 4096 → 4096       │
                 │ # actual information │
                 │   to retrieve        │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ 32 attention heads   │
                 │ 128 dims / head      │
                 │ # split attention    │
                 │   into 32 parallel   │
                 │   subspaces          │
                 └──────────┬───────────┘
                            │
                 ┌──────────┴──────────┐
                 │                     │
                 ▼                     ▼
          ┌──────────────┐      ┌──────────────┐
          │ RMSNorm(Q)   │      │ RMSNorm(K)   │
          │ # normalize  │      │ # normalize  │
          │   magnitudes │      │   magnitudes │
          │ # keeps QK   │      │ # keeps QK   │
          │   stable     │      │   stable     │
          └──────┬───────┘      └──────┬───────┘
                 │                     │
                 └──────────┬──────────┘
                            ▼
                 ┌──────────────────────┐
                 │ RoPE                 │
                 │ # inject token       │
                 │   position into Q/K  │
                 │ # for image: spatial │
                 │   location matters   │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Attention            │
                 │ softmax(QKᵀ) · V     │
                 │                      │
                 │ # each token decides │
                 │   which other tokens │
                 │   matter to it       │
                 │ # image tokens can   │
                 │   read text/context  │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Output Linear        │
                 │ 4096 → 4096          │
                 │ # merge attention    │
                 │   heads back         │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ Gated Residual       │
                 │ x + gate·attention   │
                 │ # residual keeps old │
                 │   information        │
                 │ # gate controls how  │
                 │   much new info      │
                 │   gets added         │
                 └──────────┬───────────┘
                            │
                            ▼
                 ┌──────────────────────┐
                 │ LayerNorm #2         │
                 │ # stabilize again    │
                 │   before the MLP     │
                 └──────────┬───────────┘
                            │
                            │ + timestep modulation
                            ▼
            ┌────────────────────────────────────┐
            │             SwiGLU MLP             │
            │       # feed-forward network       │
            │       # each token processed       │
            │         independently              │
            │                                    │
            │               4096                 │
            │                 │                  │
            │          ┌──────┴──────┐           │
            │          │             │           │
            │          ▼             ▼           │
            │   Linear 4096→12288  Linear        │
            │   # expansion        4096→12288    │
            │   # temporarily      # gate branch │
            │     more features                  │
            │          │             │           │
            │          │           SiLU          │
            │          │           # nonlinear   │
            │          │             gate        │
            │          │             │           │
            │          └──────×──────┘           │
            │                 │                  │
            │                 ▼                  │
            │        gated 12288-D features      │
            │        # gate keeps useful         │
            │          features and suppresses   │
            │          others                    │
            │                 │                  │
            │                 ▼                  │
            │        Linear 12288 → 4096         │
            │        # compress back to          │
            │          model hidden size         │
            └─────────────────┬──────────────────┘
                              │
                              ▼
                   ┌──────────────────────┐
                   │ Gated Residual       │
                   │ x + gate·MLP         │
                   │ # keep old features  │
                   │   + selected new     │
                   │   transformed ones   │
                   └──────────┬───────────┘
                              │
                              │ repeat ×32
                              ▼
                   ┌──────────────────────┐
                   │ norm_out             │
                   │ AdaLayerNorm         │
                   │ # final normalized   │
                   │   representation     │
                   │ # still conditioned  │
                   │   by timestep        │
                   └──────────┬───────────┘
                              │
                              ▼
                   ┌──────────────────────┐
                   │ proj_out             │
                   │ Linear 4096 → 64     │
                   │ # return from        │
                   │   transformer space  │
                   │   to VAE    space    │
                   └──────────┬───────────┘
                              │
                              ▼
                   FLOW / VELOCITY PRED.
                     vθ(x_t, t, cond)
                   # not the final image
                   # predicts how x_t
                   # should change
                              │
                              ▼
                   ┌──────────────────────┐
                   │ Flow Scheduler       │
                   │ x_t → x_(t-Δt)       │
                   │ # applies predicted  │
                   │   update to latent   │
                   └──────────┬───────────┘
                              │
                              │ repeat many timesteps
                              │
              ┌───────────────┴─────────────────┐
              │                                 │
          early steps                       late steps
              │                                 │
        x_t ≈ pure noise              x_t ≈ structured latent
              │                                 │
              └───────────────┬─────────────────┘
                              ▼
                        clean latent x_0
                        # no longer noise;
                        # compressed image
                              │
                              ▼
                        VAE Decoder
                        # latent → pixels
                              │
                              ▼
                            IMAGE
```